# Running the configurations the campaign missed

This notebook is self contained. It carries the cells it needs from
`federated_medical_v4.ipynb`, so **Run All** is enough; nothing has to be pasted
anywhere.

If your main notebook is already open with its kernel alive, you do not need
this file at all. Paste `run_missing.py` into a cell below the FedMoE cell
instead, which is quicker.

**What it fills in**

| Part | What runs | Default | Time |
|---|---|---|---|
| A | FedMoE in the 25 class space: K = 5, and experts only at K = 2, 4, 6, 7, 10 | on | about 30 min |
| B | FedMoE in the 2 class space: K = 2, 4, 5, 7 | on | about 15 min |
| C | K = 8 and K = 9 | off | about 15 min |
| D | Extra seeds for every method | off | about 50 min per seed per space |

Part A is the one that matters. It turns the experts only result from a single
measurement at K = 3 into a curve across six values of K, which is what decides
whether that result can be reported as a finding or only as a hypothesis.

Part D cannot run from this notebook, because the baseline runners it needs
(`run_centralized`, `run_local_only`) were not copied here. Leave it off, or run
it from the main notebook.

**Before you start**: put `results.json` next to this notebook. Every
configuration already in it is skipped, so the notebook is safe to run again
after an interrupted session.

**When it finishes**: send `results.json` back. Every table and figure in the
thesis is generated from that file.


## 1. Setup: imports and seeding

In [1]:
import sys
!{sys.executable} -m pip install -q torch torchvision medmnist scikit-learn
import os, time, json, copy, random, itertools
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset, ConcatDataset
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, roc_auc_score)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
import warnings; warnings.filterwarnings('ignore')

from medmnist import (PneumoniaMNIST, BreastMNIST, DermaMNIST, PathMNIST,
                      RetinaMNIST, BloodMNIST)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

print('Device :', DEVICE, '| PyTorch', torch.__version__)
if DEVICE.type == 'cpu':
    print('Running on CPU. Enable a GPU runtime before running the full grid.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.6 MB/s eta 0:00:00
Device : cuda | PyTorch 2.11.0+cu130


## 2. Configuration, RESULTS/HISTORY, record/tag_rows

In [2]:
CONFIG = {
    'num_rounds'          : 8,
    'local_epochs'        : 4,
    'batch_size'          : 64,
    'learning_rate'       : 0.0005,
    'image_size'          : 96,
    'mu_prox'             : 0.01,
    'server_lr'           : 0.1,
    'beta1'               : 0.9,
    'beta2'               : 0.99,
    'tau'                 : 1e-3,
    'max_train_per_client': 2000,
    'max_val_per_client'  : 500,
    'max_test_per_client' : 1000,
    'label_space'         : 'union',
    'classes_per_client'  : None,
    'num_public'          : 250,
    'distill_epochs'      : 8,
    'temperature'         : 4.0,
    'seeds'               : [0],
    'new_disease'         : 'BloodMNIST',
    'new_disease_epochs'  : 10,
    'new_disease_train'   : 1500,
    # Q4: how many experts. Five clients with top-1 routing can occupy at most five experts, so
    # the counts above five are the test of whether extra capacity is used or wasted.
    'expert_counts'       : [2, 3, 4, 6, 7, 10],
    'expert_counts_top2'  : [6, 10],
    'expert_counts_binary': [3, 6, 10],
    # Q5/Q6: fractions of the held-out client's own training data. 0.0 is the zero-shot point.
    'bootstrap_fractions' : [0.0, 0.1, 0.25, 0.5, 1.0],
    'bootstrap_algo'      : 'fedadam',
}

# Stages. Each one is roughly self-contained and writes its results to disk when it finishes, so
# the notebook can be run in parts and resumed with load_results() after a disconnect.
#
#   core      ~55 min   Q1, Q3   baselines, four federated algorithms, granularity sweep
#   fair      ~27 min   Q2       matched sample exposure
#   privacy   ~30 min   Q8       clipping and noise
#   moe       ~40 min   Q4       expert sweep in the superset label space
#   bootstrap ~45 min   Q5, Q6   leave-one-client-out in the superset label space
#   binary    ~85 min   Q7       the whole campaign again in healthy/diseased
#   extras    ~35 min   Q9-Q11   scalability, seeds, representation, explainability
#   export     ~1 min            tables.tex, figures and results files for the report
STAGES = {
    'core'     : True,
    'fair'     : True,
    'privacy'  : True,
    'moe'      : True,
    'bootstrap': True,
    'binary'   : True,
    'extras'   : True,
    'export'   : True,
}

RUN = {
    'centralized'      : STAGES['core'],
    'local_only'       : STAGES['core'],
    'federated'        : STAGES['core'],
    'distillation'     : STAGES['core'],
    'multiclass_sweep' : STAGES['core'],
    'distill_ablation' : STAGES['core'],
    'privacy'          : STAGES['privacy'],
    'new_disease'      : STAGES['bootstrap'],
    'explainability'   : STAGES['extras'],
    'representation'   : STAGES['extras'],
    'fair_protocol'    : STAGES['fair'],
    'moe_mix'          : STAGES['moe'],
    'bootstrap'        : STAGES['bootstrap'],
    'zero_shot_ablation': STAGES['bootstrap'],
    'scalability'      : STAGES['extras'],
    'reliability'      : STAGES['extras'],
    'binary_campaign'  : STAGES['binary'],
}

BINARY_MAPS = {
    'PneumoniaMNIST' : {0: 0, 1: 1},
    'BreastMNIST'    : {0: 1, 1: 0},
    'DermaMNIST'     : {0: 1, 1: 1, 2: 0, 3: 0, 4: 1, 5: 0, 6: 0},
    'PathMNIST'      : {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 1, 8: 1},
    'RetinaMNIST'    : {0: 0, 1: 1, 2: 1, 3: 1, 4: 1},
}

DATASET_LOADERS = {
    'PneumoniaMNIST' : PneumoniaMNIST,
    'BreastMNIST'    : BreastMNIST,
    'DermaMNIST'     : DermaMNIST,
    'PathMNIST'      : PathMNIST,
    'RetinaMNIST'    : RetinaMNIST,
}

MODALITY = {
    'PneumoniaMNIST' : 'chest X-ray',
    'BreastMNIST'    : 'breast ultrasound',
    'DermaMNIST'     : 'dermatoscopy',
    'PathMNIST'      : 'histopathology',
    'RetinaMNIST'    : 'fundus photography',
}

RESULTS = []
HISTORY = []

def record(store, **kw):
    store.append(kw)
    return kw

def _fmt(v, w=7, nd=4):
    """Format a metric that may legitimately be missing. AUC is undefined for a client whose
    test split does not contain every one of its classes, so it can be None."""
    return f'{v:{w}.{nd}f}' if isinstance(v, (int, float)) else f'{"-":>{w}}'

def tag_rows(start_index, **kw):
    """Label every RESULTS row appended since start_index. Used to mark which campaign a row
    belongs to without changing the signature of every run_* function."""
    for r in RESULTS[start_index:]:
        r.update(kw)


## 3. Persistence: load results.json and resume

In [3]:
import gc
# Where results are stored depends on the platform. The notebook is run on Colab and on Kaggle,
# which have different persistent locations, so it detects which one it is on rather than being
# edited between them.
if os.path.isdir('/kaggle'):
    # Kaggle: /kaggle/working survives the session and is downloadable from the Output tab.
    SAVE_DIR = '/kaggle/working'
    PLATFORM = 'kaggle'
elif os.path.isdir('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/fl_thesis'
    PLATFORM = 'colab'
else:
    SAVE_DIR = os.path.abspath('fl_thesis')
    PLATFORM = 'local'
print(f'platform: {PLATFORM} | results directory: {SAVE_DIR}')
RESULTS_FILE = os.path.join(SAVE_DIR, 'results.json')
os.makedirs(SAVE_DIR, exist_ok=True)

def save_results(path=None):
    tmp = RESULTS_FILE + '.tmp'
    with open(tmp, 'w') as f:
        json.dump({'config': CONFIG, 'results': RESULTS, 'history': HISTORY}, f,
                  indent=1, default=str)
    os.replace(tmp, RESULTS_FILE)
    print(f'  [saved {len(RESULTS)} rows to Drive]')

def load_results(path=None):
    """Pull previously saved rows back into memory. Rows already present are not duplicated, so
    this is safe to call more than once."""
    if not os.path.exists(RESULTS_FILE) or os.path.getsize(RESULTS_FILE) == 0:
        print('no previous results on Drive - starting fresh')
        return
    try:
        blob = json.load(open(RESULTS_FILE))
    except json.JSONDecodeError:
        print('saved file is unreadable - starting fresh')
        return
    # The key has to separate every run that can legitimately share a setting. The
    # matched-exposure sweep, for instance, writes six FedAvg runs under setting='fair_protocol'
    # that differ only in local_epochs and fair_rounds; without those fields five of the six are
    # discarded as duplicates on reload.
    key = lambda r: (r.get('experiment'), r.get('method'), r.get('client'), r.get('seed'),
                     r.get('setting'), r.get('fraction'), r.get('num_experts'), r.get('init'),
                     r.get('local_epochs'), r.get('fair_rounds'), r.get('top_k'),
                     r.get('n_public'), r.get('sync_clients'), r.get('dp_clip'),
                     r.get('dp_noise'), r.get('classes_per_client'))
    have = {key(r) for r in RESULTS}
    added = 0
    for r in blob.get('results', []):
        if key(r) not in have:
            RESULTS.append(r)
            have.add(key(r))
            added += 1
    for h in blob.get('history', []):
        HISTORY.append(h)
    print(f'loaded {added} rows from Drive (total {len(RESULTS)})')

def already_done(**crit):
    """True when a run matching these fields is already recorded. Every guard in the notebook
    goes through this, so a re-run continues rather than repeating finished work."""
    return any(all(r.get(k) == v for k, v in crit.items()) for r in RESULTS)

def gpu_report(tag=''):
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print(f'  [{tag} GPU free {free / 1e9:.2f} GB of {total / 1e9:.2f} GB]')

def release(obj, depth=0):
    """Drop the model objects a finished run still holds and hand the memory back to the GPU.
    Each run keeps five client models and a global one with their optimizer state, roughly 0.8 GB;
    a dozen of them exhausts a 15 GB card. The metrics are already in RESULTS, so nothing measured
    is lost."""
    if isinstance(obj, dict) and depth <= 3:
        if {'locals', 'global', 'model'} & set(obj):
            for k in ('locals', 'global', 'model'):
                obj.pop(k, None)
        else:
            for v in list(obj.values()):
                release(v, depth + 1)
    if depth == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def checkpoint(start, campaign=None, run=None):
    """Called after every individual experiment: label the new rows, write them to Drive, and
    free the weights."""
    if campaign:
        for r in RESULTS[start:]:
            r['campaign'] = campaign
    save_results()
    if run is not None:
        release(run)
    return len(RESULTS)

load_results()
print('GPU available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then run this cell again.')


platform: kaggle | results directory: /kaggle/working
no previous results on Drive - starting fresh
GPU available: True


## 4. Data: make_clients, head_size, CLIENTS

In [4]:
imagenet_mean, imagenet_std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
transform_test = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

class MappedDataset(Dataset):
    def __init__(self, base, indices, label_map, offset=0):
        self.base = base
        self.indices = indices
        self.label_map = label_map
        self.offset = offset
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, lab = self.base[self.indices[i]]
        native = int(np.array(lab).reshape(-1)[0])
        return img, self.label_map[native] + self.offset

def dataset_labels(ds):
    return np.array(ds.labels).reshape(-1)

def make_label_map(name, y, label_space, k):
    if label_space == 'binary':
        return BINARY_MAPS[name]
    classes = sorted(np.unique(y).tolist())
    if k is not None:
        counts = {c: int((y == c).sum()) for c in classes}
        classes = sorted(sorted(classes, key=lambda c: -counts[c])[:k])
    return {c: i for i, c in enumerate(classes)}

def build_split(loader_class, split, transform, cap, label_map, seed):
    base = loader_class(split=split, transform=transform, download=True)
    y = dataset_labels(base)
    keep = np.where(np.isin(y, list(label_map)))[0]
    if cap is not None and len(keep) > cap:
        keep = np.random.RandomState(seed).choice(keep, cap, replace=False)
    return MappedDataset(base, keep.tolist(), label_map)

def make_clients(label_space=None, classes_per_client=None, seed=0):
    label_space = label_space or CONFIG['label_space']
    if classes_per_client is None:
        classes_per_client = CONFIG['classes_per_client']
    names = list(DATASET_LOADERS)
    train, val, test, num_classes = [], [], [], []
    for name in names:
        loader_class = DATASET_LOADERS[name]
        probe = loader_class(split='train', download=True)
        label_map = make_label_map(name, dataset_labels(probe), label_space, classes_per_client)
        train.append(build_split(loader_class, 'train', transform_train,
                                 CONFIG['max_train_per_client'], label_map, seed))
        val.append(build_split(loader_class, 'val', transform_test,
                               CONFIG['max_val_per_client'], label_map, seed))
        test.append(build_split(loader_class, 'test', transform_test,
                                CONFIG['max_test_per_client'], label_map, seed))
        num_classes.append(len(set(label_map.values())))

    offsets = [0] * len(names)
    if label_space == 'union':
        offsets = np.cumsum([0] + num_classes[:-1]).tolist()
        for off, tr, va, te in zip(offsets, train, val, test):
            tr.offset = va.offset = te.offset = off

    class_ids = [list(range(o, o + n)) for o, n in zip(offsets, num_classes)]
    total_classes = sum(num_classes) if label_space == 'union' else max(num_classes)

    return {
        'names'        : names,
        'train'        : train,
        'val'          : val,
        'test'         : test,
        'num_classes'  : num_classes,
        'class_ids'    : class_ids,
        'total_classes': total_classes,
        'label_space'  : label_space,
        'sizes'        : [len(d) for d in train],
        'train_loaders': [DataLoader(d, batch_size=CONFIG['batch_size'], shuffle=True) for d in train],
        'test_loaders' : [DataLoader(d, batch_size=128, shuffle=False) for d in test],
    }

def head_size(clients, i):
    return clients['total_classes'] if clients['label_space'] == 'union' else clients['num_classes'][i]

def label_counts(ds, num_classes):
    y = dataset_labels(ds.base)[ds.indices]
    mapped = np.array([ds.label_map[int(v)] for v in y])
    return np.bincount(mapped, minlength=num_classes)

def describe(clients):
    rows = []
    for i, name in enumerate(clients['names']):
        counts = label_counts(clients['train'][i], clients['num_classes'][i])
        rows.append({
            'client': name,
            'modality': MODALITY[name],
            'classes': clients['num_classes'][i],
            'train': len(clients['train'][i]),
            'val': len(clients['val'][i]),
            'test': len(clients['test'][i]),
            'imbalance': round(float(counts.max() / max(counts.min(), 1)), 2),
        })
    return rows

CLIENTS = make_clients()
TABLE1 = describe(CLIENTS)
for r in TABLE1:
    print(r)
print('total classes in shared output space:', CLIENTS['total_classes'])

100%|██████████| 4.17M/4.17M [00:06<00:00, 661kB/s]
100%|██████████| 560k/560k [00:01<00:00, 504kB/s]
100%|██████████| 19.7M/19.7M [00:15<00:00, 1.30MB/s]
100%|██████████| 206M/206M [02:10<00:00, 1.58MB/s]
100%|██████████| 3.29M/3.29M [00:04<00:00, 716kB/s]


{'client': 'PneumoniaMNIST', 'modality': 'chest X-ray', 'classes': 2, 'train': 2000, 'val': 500, 'test': 624, 'imbalance': 2.8}
{'client': 'BreastMNIST', 'modality': 'breast ultrasound', 'classes': 2, 'train': 546, 'val': 78, 'test': 156, 'imbalance': 2.71}
{'client': 'DermaMNIST', 'modality': 'dermatoscopy', 'classes': 7, 'train': 2000, 'val': 500, 'test': 1000, 'imbalance': 58.52}
{'client': 'PathMNIST', 'modality': 'histopathology', 'classes': 9, 'train': 2000, 'val': 500, 'test': 1000, 'imbalance': 1.61}
{'client': 'RetinaMNIST', 'modality': 'fundus photography', 'classes': 5, 'train': 1080, 'val': 120, 'test': 400, 'imbalance': 7.36}
total classes in shared output space: 25


## 5. Evaluation helpers

In [5]:
def collect_predictions(model, loader, device=DEVICE):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            out = model(imgs.to(device))
            ps.append(F.softmax(out, dim=1).cpu().numpy())
            ys.append(labels.numpy())
    return np.concatenate(ys), np.concatenate(ps)

def safe_auc(y_local, p_local):
    n = p_local.shape[1]
    try:
        if n == 2:
            return float(roc_auc_score(y_local, p_local[:, 1]))
        if len(np.unique(y_local)) < n:
            return None
        return float(roc_auc_score(y_local, p_local, multi_class='ovr',
                                   average='macro', labels=list(range(n))))
    except ValueError:
        return None

def compute_metrics(y, prob, class_ids):
    ids = np.array(class_ids)
    pred = ids[prob[:, ids].argmax(1)]
    acc = accuracy_score(y, pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y, pred, labels=ids,
                                                    average='macro', zero_division=0)
    _, per_class_recall, _, support = precision_recall_fscore_support(
        y, pred, labels=ids, average=None, zero_division=0)
    p_local = prob[:, ids]
    p_local = p_local / p_local.sum(1, keepdims=True).clip(1e-12)
    y_local = np.searchsorted(ids, y)
    out = {
        'accuracy'  : float(acc),
        'precision' : float(pr),
        'recall'    : float(rc),
        'f1'        : float(f1),
        'auc'       : safe_auc(y_local, p_local),
        'per_class_recall': [round(float(v), 4) for v in per_class_recall],
        'support'   : [int(v) for v in support],
        'confusion' : confusion_matrix(y, pred, labels=ids).tolist(),
    }
    if prob.shape[1] > len(ids):
        open_pred = prob.argmax(1)
        out['open_accuracy'] = float((open_pred == y).mean())
        out['outside_rate'] = float((~np.isin(open_pred, ids)).mean())
    return out

def evaluate_client(model, loader, class_ids):
    y, prob = collect_predictions(model, loader)
    return compute_metrics(y, prob, class_ids)

def mean_over_clients(metric_list, key):
    vals = [m[key] for m in metric_list if m.get(key) is not None]
    return float(np.mean(vals)) if vals else None

## 6. Model and state helpers

In [6]:
class FederatedModel(nn.Module):
    def __init__(self, num_classes, pretrained=True, freeze_stem=True):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet18(weights=weights)
        backbone.fc = nn.Identity()
        self.backbone = backbone
        if freeze_stem:
            for p in self.backbone.conv1.parameters():
                p.requires_grad = False
            for p in self.backbone.bn1.parameters():
                p.requires_grad = False
        self.head = nn.Sequential(
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    def features(self, x):
        return self.backbone(x)
    def forward(self, x):
        return self.head(self.backbone(x))

def bn_param_names(model):
    names = set()
    for name, module in model.named_modules():
        if isinstance(module, nn.BatchNorm2d):
            names.add(name + '.weight')
            names.add(name + '.bias')
    return names

def federated_names(model, local_head=False):
    bn = bn_param_names(model)
    out = []
    for n, p in model.named_parameters():
        if not p.requires_grad or n in bn:
            continue
        if local_head and n.startswith('head.'):
            continue
        out.append(n)
    return out

def extract_state(model, names):
    params = dict(model.named_parameters())
    return {n: params[n].detach().clone() for n in names}

def load_state(model, state):
    full = model.state_dict()
    for k, v in state.items():
        full[k] = v
    model.load_state_dict(full)

def state_bytes(state):
    return sum(v.numel() * v.element_size() for v in state.values())

_probe = FederatedModel(CLIENTS['total_classes'])
print('total parameters     :', f'{sum(p.numel() for p in _probe.parameters()):,}')
print('trainable            :', f'{sum(p.numel() for p in _probe.parameters() if p.requires_grad):,}')
print('federated (union)    :', f'{sum(_probe.state_dict()[n].numel() for n in federated_names(_probe)):,}')
print('federated (native)   :', f'{sum(_probe.state_dict()[n].numel() for n in federated_names(_probe, local_head=True)):,}')
print('kept local (BN)      :', f'{sum(_probe.state_dict()[n].numel() for n in bn_param_names(_probe)):,}')
del _probe

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 170MB/s]


total parameters     : 11,245,401
trainable            : 11,235,865
federated (union)    : 11,226,393
federated (native)   : 11,157,504
kept local (BN)      : 9,600


## 7. Local training and communication accounting

In [7]:
def train_local(model, loader, optimizer, epochs, global_params=None, mu=0.0, device=DEVICE):
    model.train()
    ce = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = ce(model(imgs), labels)
            if global_params is not None and mu > 0:
                prox = sum(((p - global_params[n]) ** 2).sum()
                           for n, p in model.named_parameters() if n in global_params)
                loss = loss + (mu / 2) * prox
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

CLIP_NORMS = []

def privatize(model, global_state, clip_norm, noise_multiplier, device=DEVICE):
    params = dict(model.named_parameters())
    delta = {n: params[n].detach() - global_state[n] for n in global_state}
    total = torch.sqrt(sum((d.float() ** 2).sum() for d in delta.values()))
    scale = min(1.0, clip_norm / (total.item() + 1e-12))
    new_state = {}
    for n, d in delta.items():
        d = d * scale
        if noise_multiplier > 0:
            d = d + torch.normal(0.0, noise_multiplier * clip_norm, size=d.shape, device=device)
        new_state[n] = global_state[n] + d
    load_state(model, new_state)
    CLIP_NORMS.append(float(total.item()))
    return float(total.item())

class Comm:
    def __init__(self):
        self.up = 0
        self.down = 0
    def send(self, nbytes, n=1):
        self.up += nbytes * n
    def receive(self, nbytes, n=1):
        self.down += nbytes * n
    def total_mb(self):
        return (self.up + self.down) / 1e6
    def summary(self, rounds, n_clients):
        return {
            'up_mb': self.up / 1e6,
            'down_mb': self.down / 1e6,
            'total_mb': self.total_mb(),
            'mb_per_client_per_round': self.total_mb() / max(rounds * n_clients, 1),
            'rounds': rounds,
        }

## 8. aggregate()

In [8]:
def aggregate(states, sizes):
    total = sum(sizes)
    keys = list(states[0])
    return {k: sum(states[i][k].float() * (sizes[i] / total) for i in range(len(states))) for k in keys}

def run_federated(clients, algo='fedavg', seed=0, rounds=None, dp=None, setting='main', verbose=True):
    set_seed(seed)
    rounds = rounds or CONFIG['num_rounds']
    local_head = clients['label_space'] == 'native'
    n = len(clients['names'])

    locals_ = [FederatedModel(head_size(clients, i)).to(DEVICE) for i in range(n)]
    opts = [optim.Adam(filter(lambda p: p.requires_grad, m.parameters()), lr=CONFIG['learning_rate'])
            for m in locals_]
    global_model = FederatedModel(head_size(clients, 0)).to(DEVICE)
    fed_names = federated_names(global_model, local_head=local_head)
    comm = Comm()
    server_m, server_v = {}, {}
    label = {'fedavg': 'FedAvg', 'fedprox': 'FedProx', 'fedadam': 'FedAdam'}[algo]
    if dp is not None:
        label += f" (DP C={dp['clip']} s={dp['noise']})"

    t0 = time.time()
    for rnd in range(1, rounds + 1):
        g_state = extract_state(global_model, fed_names)
        nbytes = state_bytes(g_state)
        for m in locals_:
            load_state(m, g_state)
            comm.receive(nbytes)

        for i, m in enumerate(locals_):
            train_local(m, clients['train_loaders'][i], opts[i], CONFIG['local_epochs'],
                        global_params=g_state if algo == 'fedprox' else None,
                        mu=CONFIG['mu_prox'] if algo == 'fedprox' else 0.0)
            if dp is not None:
                privatize(m, g_state, dp['clip'], dp['noise'])

        client_states = [extract_state(m, fed_names) for m in locals_]
        for s in client_states:
            comm.send(state_bytes(s))
        avg = aggregate(client_states, clients['sizes'])

        if algo == 'fedadam':
            new_state = {}
            for k in avg:
                delta = avg[k] - g_state[k].float()
                if k not in server_m:
                    server_m[k] = torch.zeros_like(delta)
                    server_v[k] = torch.full_like(delta, CONFIG['tau'] ** 2)
                server_m[k] = CONFIG['beta1'] * server_m[k] + (1 - CONFIG['beta1']) * delta
                server_v[k] = CONFIG['beta2'] * server_v[k] + (1 - CONFIG['beta2']) * delta ** 2
                new_state[k] = (g_state[k].float() + CONFIG['server_lr'] * server_m[k]
                                / (torch.sqrt(server_v[k]) + CONFIG['tau']))
            load_state(global_model, new_state)
        else:
            load_state(global_model, avg)

        metrics = [evaluate_client(m, ld, ids)
                   for m, ld, ids in zip(locals_, clients['test_loaders'], clients['class_ids'])]
        for name, mt in zip(clients['names'], metrics):
            record(HISTORY, method=label, algo=algo, seed=seed, round=rnd, client=name,
                   label_space=clients['label_space'], accuracy=mt['accuracy'], f1=mt['f1'],
                   cum_mb=comm.total_mb(), elapsed_s=time.time() - t0)
        if verbose:
            accs = ' | '.join(f'{nm[:5]} {mt["accuracy"]:.3f}' for nm, mt in zip(clients['names'], metrics))
            print(f'  {label} r{rnd:02d} | mean acc {mean_over_clients(metrics, "accuracy"):.4f} | '
                  f'mean f1 {mean_over_clients(metrics, "f1"):.4f} | {accs}')

    elapsed = time.time() - t0
    summary = comm.summary(rounds, n)
    for i, (name, mt) in enumerate(zip(clients['names'], metrics)):
        record(RESULTS, experiment='federated' if dp is None else 'privacy', method=label, algo=algo,
               client=name, seed=seed, label_space=clients['label_space'],
               classes=clients['num_classes'][i], setting=setting, time_s=elapsed,
               dp_clip=None if dp is None else dp['clip'],
               dp_noise=None if dp is None else dp['noise'],
               up_mb=summary['up_mb'], down_mb=summary['down_mb'], rounds=rounds,
               **{k: v for k, v in mt.items() if k not in ('confusion', 'per_class_recall', 'support')})
    print(f'{label} | mean acc {mean_over_clients(metrics, "accuracy"):.4f} | '
          f'mean F1 {mean_over_clients(metrics, "f1"):.4f} | {summary["total_mb"]:.1f} MB | {elapsed:.0f}s')
    return {'locals': locals_, 'global': global_model, 'metrics': metrics,
            'comm': summary, 'time_s': elapsed, 'label': label}

## 9. FedMoE definitions only (driver removed)

In [9]:
class SparseMoEHead(nn.Module):
    def __init__(self, in_dim=512, hidden=128, num_experts=3, top_k=1):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.gate = nn.Linear(in_dim, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.3))
            for _ in range(num_experts)
        ])

    def forward(self, feats, return_gate=False):
        # Switch-Transformer style routing: softmax over ALL experts first, then keep the top-k
        # probability without renormalizing. Taking topk on the raw logits and softmaxing those
        # gives a constant 1.0 when top_k == 1, so no gradient reaches self.gate and the routing
        # never leaves its random initialization.
        gate_logits = self.gate(feats)
        probs = F.softmax(gate_logits, dim=1)
        topk_val, topk_idx = probs.topk(self.top_k, dim=1)
        sparse_w = torch.zeros_like(probs).scatter(1, topk_idx, topk_val)
        expert_out = torch.stack([e(feats) for e in self.experts], dim=1)
        mixed = (sparse_w.unsqueeze(-1) * expert_out).sum(dim=1)
        if return_gate:
            return mixed, sparse_w
        return mixed

class FedMoEModel(nn.Module):
    def __init__(self, num_classes, num_experts=3, top_k=1, pretrained=True, freeze_stem=True):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet18(weights=weights)
        backbone.fc = nn.Identity()
        self.backbone = backbone
        if freeze_stem:
            for p in self.backbone.conv1.parameters():
                p.requires_grad = False
            for p in self.backbone.bn1.parameters():
                p.requires_grad = False
        self.moe = SparseMoEHead(in_dim=512, hidden=128, num_experts=num_experts, top_k=top_k)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x, return_gate=False):
        feats = self.backbone(x)
        mixed, gate_weights = self.moe(feats, return_gate=True)
        out = self.classifier(mixed)
        if return_gate:
            return out, gate_weights
        return out

def fedmoe_federated_names(model, share_backbone=True, share_head=True):
    # The gate stays local (that is the personalization). With share_backbone=True the backbone and
    # the final classifier travel exactly as in FedAvg, so FedMoE differs from FedAvg by the MoE
    # mechanism alone. share_backbone=False is the ablation where only the experts travel.
    bn = bn_param_names(model)
    out = []
    for n, p in model.named_parameters():
        if not p.requires_grad or n in bn or n.startswith('moe.gate'):
            continue
        if n.startswith('moe.experts.'):
            out.append(n)
        elif share_backbone and n.startswith('backbone.'):
            out.append(n)
        elif share_backbone and share_head and n.startswith('classifier.'):
            out.append(n)
    return out

def run_fedmoe(clients, num_experts=3, top_k=1, seed=0, rounds=None, share_backbone=True,
               setting='fedmoe'):
    set_seed(seed)
    rounds = rounds or CONFIG['num_rounds']
    n = len(clients['names'])

    locals_ = [FedMoEModel(head_size(clients, i), num_experts=num_experts, top_k=top_k).to(DEVICE)
               for i in range(n)]
    opts = [optim.Adam(filter(lambda p: p.requires_grad, m.parameters()), lr=CONFIG['learning_rate'])
            for m in locals_]
    global_model = FedMoEModel(head_size(clients, 0), num_experts=num_experts, top_k=top_k).to(DEVICE)
    fed_names = fedmoe_federated_names(global_model, share_backbone=share_backbone,
                                       share_head=(clients['label_space'] != 'native'))
    comm = Comm()
    label = f'FedMoE (K={num_experts},top{top_k},bb={int(share_backbone)})'

    t0 = time.time()
    metrics = None
    for rnd in range(1, rounds + 1):
        g_state = extract_state(global_model, fed_names)
        nbytes = state_bytes(g_state)
        for m in locals_:
            load_state(m, g_state)
            comm.receive(nbytes)

        for i, m in enumerate(locals_):
            train_local(m, clients['train_loaders'][i], opts[i], CONFIG['local_epochs'])

        client_states = [extract_state(m, fed_names) for m in locals_]
        for s in client_states:
            comm.send(state_bytes(s))
        load_state(global_model, aggregate(client_states, clients['sizes']))

        metrics = [evaluate_client(m, ld, ids)
                   for m, ld, ids in zip(locals_, clients['test_loaders'], clients['class_ids'])]
        for name, mt in zip(clients['names'], metrics):
            record(HISTORY, method=label, algo='fedmoe', seed=seed, round=rnd, client=name,
                   label_space=clients['label_space'], accuracy=mt['accuracy'], f1=mt['f1'],
                   cum_mb=comm.total_mb(), elapsed_s=time.time() - t0)
        print(f'  {label} r{rnd:02d} | mean acc {mean_over_clients(metrics, "accuracy"):.4f} | '
              f'mean f1 {mean_over_clients(metrics, "f1"):.4f}')

    elapsed = time.time() - t0
    summary = comm.summary(rounds, n)

    # Q4 asks how many experts are useful, so the routing itself has to be measured and not just
    # the accuracy. Three quantities are recorded: the mean gate vector per client, the share of
    # test images each expert actually receives across the whole federation, and the entropy of
    # each client's gate. An expert that receives less than 1 % of the traffic is counted as idle:
    # with five clients and top-1 routing at most five experts can ever be occupied, so a large K
    # is expected to leave experts idle and the measurement shows whether that is what happens.
    gate_table = []
    usage_counts = torch.zeros(num_experts)
    total_images = 0
    for i, (name, m) in enumerate(zip(clients['names'], locals_)):
        m.eval()
        weights_sum, count = torch.zeros(num_experts), 0
        client_counts = torch.zeros(num_experts)
        with torch.no_grad():
            for imgs, _ in clients['test_loaders'][i]:
                _, gw = m(imgs.to(DEVICE), return_gate=True)
                weights_sum += gw.sum(dim=0).cpu()
                picks = gw.argmax(dim=1).cpu()
                client_counts += torch.bincount(picks, minlength=num_experts).float()
                count += gw.shape[0]
        mean_gate = (weights_sum / max(count, 1)).tolist()
        usage_counts += client_counts
        total_images += count
        dominant = int(np.argmax(mean_gate))
        p = np.array(mean_gate, dtype=float)
        p = p / max(p.sum(), 1e-12)
        entropy = float(-(p * np.log(p + 1e-12)).sum())
        client_share = (client_counts / max(count, 1)).tolist()
        gate_table.append({'client': name, 'modality': MODALITY[name], 'dominant_expert': dominant,
                           'gate_entropy': round(entropy, 4),
                           **{f'expert_{k}': round(mean_gate[k], 4) for k in range(num_experts)}})
        record(RESULTS, experiment='fedmoe', method=label, client=name, seed=seed,
               label_space=clients['label_space'], classes=clients['num_classes'][i],
               setting=setting, num_experts=num_experts, top_k=top_k,
               share_backbone=share_backbone, dominant_expert=dominant, gate_weights=mean_gate,
               gate_entropy=entropy, client_expert_share=[round(v, 4) for v in client_share],
               time_s=elapsed, up_mb=summary['up_mb'], down_mb=summary['down_mb'], rounds=rounds,
               **{k: v for k, v in metrics[i].items()
                  if k not in ('confusion', 'per_class_recall', 'support')})

    usage = (usage_counts / max(total_images, 1)).tolist()
    experts_used = int(sum(1 for u in usage if u >= 0.01))
    distinct_dominant = len({row['dominant_expert'] for row in gate_table})
    mean_entropy = float(np.mean([row['gate_entropy'] for row in gate_table]))
    for r in RESULTS:
        if r.get('experiment') == 'fedmoe' and r.get('setting') == setting and r.get('seed') == seed:
            r['experts_used'] = experts_used
            r['distinct_dominant'] = distinct_dominant
            r['expert_usage'] = [round(u, 4) for u in usage]
            r['mean_gate_entropy'] = mean_entropy

    print(f'{label} | mean acc {mean_over_clients(metrics, "accuracy"):.4f} | '
          f'mean f1 {mean_over_clients(metrics, "f1"):.4f} | '
          f'{summary["total_mb"]:.1f} MB | {elapsed:.0f}s')
    print(f'  experts receiving >=1% of images: {experts_used} of {num_experts} | '
          f'distinct dominant experts: {distinct_dominant} | mean gate entropy {mean_entropy:.3f}')
    print(f'  expert usage share over all clients: {[round(u, 3) for u in usage]}')
    print('  per-client gate weights (local, personalized, never shared with the server):')
    for row in gate_table:
        print('   ', row)
    return {'locals': locals_, 'global': global_model, 'metrics': metrics, 'gate_table': gate_table,
            'comm': summary, 'time_s': elapsed, 'label': label, 'experts_used': experts_used,
            'usage': usage, 'mean_gate_entropy': mean_entropy}


class MeanBatchStore:
    def __init__(self, clients, batches_per_client=5, seed=0):
        set_seed(seed)
        self.num_classes = clients['total_classes']
        self.mean_images, self.mean_labels = {}, {}
        for i, name in enumerate(clients['names']):
            loader = DataLoader(clients['train'][i], batch_size=CONFIG['batch_size'], shuffle=True)
            imgs_acc, labels_acc, n = None, None, 0
            for b, (imgs, labels) in enumerate(loader):
                if b >= batches_per_client:
                    break
                oh = F.one_hot(labels, num_classes=self.num_classes).float()
                bi, bl = imgs.mean(dim=0, keepdim=True), oh.mean(dim=0, keepdim=True)
                imgs_acc = bi if imgs_acc is None else imgs_acc + bi
                labels_acc = bl if labels_acc is None else labels_acc + bl
                n += 1
            self.mean_images[name] = (imgs_acc / max(n, 1)).squeeze(0)
            self.mean_labels[name] = (labels_acc / max(n, 1)).squeeze(0)

    def sample_other(self, exclude_name, rng):
        others = [n for n in self.mean_images if n != exclude_name]
        pick = others[rng.randrange(len(others))]
        return self.mean_images[pick], self.mean_labels[pick]

def fedmix_loss(model, imgs, labels, mean_store, client_name, num_classes, lam=0.2, rng=None):
    rng = rng or random
    ce = F.cross_entropy(model(imgs), labels)
    mean_img, mean_label = mean_store.sample_other(client_name, rng)
    mixed_imgs = (1 - lam) * imgs + lam * mean_img.unsqueeze(0).to(imgs.device)
    log_probs = F.log_softmax(model(mixed_imgs), dim=1)
    true_oh = F.one_hot(labels, num_classes=num_classes).float()
    # both halves of the FedMix target: the client's own label and the shared mean label of the
    # other client whose average image was mixed in.
    target = (1 - lam) * true_oh + lam * mean_label.unsqueeze(0).to(imgs.device)
    return ce + lam * (-(target * log_probs).sum(dim=1).mean())

def train_local_fedmix(model, loader, opt, epochs, mean_store, client_name, num_classes,
                       lam=0.2, seed=0):
    rng = random.Random(seed)
    model.train()
    for _ in range(epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            loss = fedmix_loss(model, imgs, labels, mean_store, client_name, num_classes,
                               lam=lam, rng=rng)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

def run_fedmix(clients, lam=0.2, seed=0, rounds=None, setting='fedmix'):
    if clients['label_space'] == 'native':
        raise ValueError('FedMix mixes label distributions between clients, which needs a shared '
                         'output space; run it in union or binary mode')
    set_seed(seed)
    rounds = rounds or CONFIG['num_rounds']
    n = len(clients['names'])
    num_classes = clients['total_classes']
    mean_store = MeanBatchStore(clients, seed=seed)

    locals_ = [FederatedModel(head_size(clients, i)).to(DEVICE) for i in range(n)]
    opts = [optim.Adam(filter(lambda p: p.requires_grad, m.parameters()), lr=CONFIG['learning_rate'])
            for m in locals_]
    global_model = FederatedModel(head_size(clients, 0)).to(DEVICE)
    fed_names = federated_names(global_model)
    comm = Comm()
    label = f'FedMix (lam={lam})'

    t0 = time.time()
    metrics = None
    for rnd in range(1, rounds + 1):
        g_state = extract_state(global_model, fed_names)
        nbytes = state_bytes(g_state)
        for m in locals_:
            load_state(m, g_state)
            comm.receive(nbytes)

        for i, m in enumerate(locals_):
            train_local_fedmix(m, clients['train_loaders'][i], opts[i], CONFIG['local_epochs'],
                               mean_store, clients['names'][i], num_classes, lam=lam, seed=seed + rnd)

        client_states = [extract_state(m, fed_names) for m in locals_]
        for s in client_states:
            comm.send(state_bytes(s))
        load_state(global_model, aggregate(client_states, clients['sizes']))

        metrics = [evaluate_client(m, ld, ids)
                   for m, ld, ids in zip(locals_, clients['test_loaders'], clients['class_ids'])]
        for name, mt in zip(clients['names'], metrics):
            record(HISTORY, method=label, algo='fedmix', seed=seed, round=rnd, client=name,
                   label_space=clients['label_space'], accuracy=mt['accuracy'], f1=mt['f1'],
                   cum_mb=comm.total_mb(), elapsed_s=time.time() - t0)
        print(f'  {label} r{rnd:02d} | mean acc {mean_over_clients(metrics, "accuracy"):.4f} | '
              f'mean f1 {mean_over_clients(metrics, "f1"):.4f}')

    elapsed = time.time() - t0
    summary = comm.summary(rounds, n)
    for i, (name, mt) in enumerate(zip(clients['names'], metrics)):
        record(RESULTS, experiment='fedmix', method=label, client=name, seed=seed,
               label_space=clients['label_space'], classes=clients['num_classes'][i],
               setting=setting, lam=lam, time_s=elapsed,
               up_mb=summary['up_mb'], down_mb=summary['down_mb'], rounds=rounds,
               **{k: v for k, v in mt.items() if k not in ('confusion', 'per_class_recall', 'support')})
    print(f'{label} | mean acc {mean_over_clients(metrics, "accuracy"):.4f} | '
          f'{summary["total_mb"]:.1f} MB | {elapsed:.0f}s')
    return {'locals': locals_, 'global': global_model, 'metrics': metrics, 'comm': summary,
            'time_s': elapsed, 'label': label}

def _checkpoint(start, campaign='union', run=None):
    """Label and save the rows written since `start`, then release the run's weights. Called
    after every individual run so that an interrupted sweep never loses what already finished and
    a long sweep does not accumulate a dozen models on the GPU."""
    tag_rows(start, campaign=campaign)
    save_results()
    if run is not None:
        release(run)
        gpu_report('after release:')
    return len(RESULTS)


## 10. Run the missing configurations


In [ ]:
# ============================================================================
# Runs everything listed under "What I want to run next" in the meeting report.
#
#   PART A  the missing FedMoE settings, 25-class space          8 runs
#   PART B  the missing FedMoE settings, 2-class space           4 runs
#   PART C  K = 8 and K = 9, optional                            4 runs
#   PART D  extra seeds for every method, optional              varies
#
# Paste as one cell in federated_medical_v4.ipynb, below the FedMoE cell.
# Or open fill_moe_gaps.ipynb, which already carries the cells this needs.
#
# Safe to re-run. Every configuration is skipped if it is already in RESULTS,
# so an interrupted session can just run the cell again. Results are written
# to the same results.json, so the thesis tables rebuild from it untouched.
#
# Turn the parts on and off here.
# ============================================================================

RUN_A_SUPERSET_MOE = True    # ~30 min
RUN_B_BINARY_MOE   = True    # ~15 min
RUN_C_K8_K9        = False   # ~15 min, low information: see the report
RUN_D_SEEDS        = False   # ~50 min per extra seed per label space

SEEDS_TO_ADD   = [1, 2]      # seed 0 already exists for most methods
SEED_SPACES    = ['union']   # add 'binary' to repeat that campaign too

# (K, top_k, share_backbone, name)
A_CONFIGS = [
    (5,  1, True,  'K5'),         # K = number of hospitals, the key missing point
    (5,  1, False, 'K5_nobb'),
    (2,  1, False, 'K2_nobb'),    # experts-only across K: the promising thread
    (4,  1, False, 'K4_nobb'),
    (6,  1, False, 'K6_nobb'),
    (7,  1, False, 'K7_nobb'),
    (10, 1, False, 'K10_nobb'),
    (5,  2, True,  'K5_top2'),
]
B_CONFIGS = [(2, 1, True, 'K2'), (4, 1, True, 'K4'),
             (5, 1, True, 'K5'), (7, 1, True, 'K7')]
C_CONFIGS = [(8, 1, True, 'K8'), (9, 1, True, 'K9'),
             (8, 1, False, 'K8_nobb'), (9, 1, False, 'K9_nobb')]

# ---------------------------------------------------------------------------
# Preflight: everything here comes from the notebook above, so check it all
# first. Running out of a name halfway through an hour-long job is worse than
# not starting.
# ---------------------------------------------------------------------------
_needed = {
    'RESULTS':      'the setup cell near the top',
    'save_results': 'the setup cell near the top',
    'tag_rows':     'the setup cell near the top',
    'release':      'the setup cell near the top',
    'gpu_report':   'the setup cell near the top',
}
if RUN_A_SUPERSET_MOE or RUN_B_BINARY_MOE or RUN_C_K8_K9:
    _needed['run_fedmoe'] = 'the FedMoE cell'
if RUN_D_SEEDS:
    _needed['run_federated'] = 'the federated algorithms cell'
    _needed['run_centralized'] = 'the baselines cell'
    _needed['run_local_only'] = 'the baselines cell'
_missing = {n: w for n, w in _needed.items() if n not in globals()}
if 'CLIENTS' not in globals() and 'make_clients' not in globals():
    _missing['make_clients'] = 'the data cell'
if _missing:
    raise NameError('\n'.join(
        ['This cell needs the notebook to have been run above it.', '',
         'Not defined in this kernel:']
        + [f'  - {n}: comes from {w}' for n, w in _missing.items()]
        + ['', 'Fix: Run All, or run every cell from the top down to and including',
           'the FedMoE cell, then run this one. The training cells skip anything',
           'already in results.json, so re-running costs little.']))


def _clients(space):
    """The clients for one label space, built only if not already in memory."""
    if space == 'union':
        if 'CLIENTS' in globals():
            return CLIENTS
        print('building superset clients')
        return make_clients()
    if 'BIN_CLIENTS' in globals():
        return BIN_CLIENTS
    print('building binary clients')
    return make_clients(label_space='binary')


def _done(**crit):
    return any(all(r.get(k) == v for k, v in crit.items()) for r in RESULTS)


_start = len(RESULTS)
_ran, _skipped = [], []


def _step(label, campaign, fn, **crit):
    if _done(**crit):
        _skipped.append(label)
        print(f'  {label}: already present, skipping')
        return
    print(f'\n--- {label} ---')
    result = fn()
    tag_rows(_start, campaign=campaign)
    save_results()           # write before releasing, so a crash costs one run
    release(result)
    gpu_report('after release:')
    _ran.append(label)


def _moe_block(configs, space, prefix, campaign, title):
    cl = _clients(space)
    print(f'\n{"=" * 70}\n{title}: {len(configs)} configurations\n{"=" * 70}')
    for K, top_k, share_bb, name in configs:
        setting = f'{prefix}{name}'
        shared = 'backbone + experts' if share_bb else 'experts only'
        _step(f'{setting}  (K={K}, top-{top_k}, {shared})', campaign,
              lambda k=K, t=top_k, b=share_bb, s=setting:
                  run_fedmoe(cl, num_experts=k, top_k=t, seed=0,
                             share_backbone=b, setting=s),
              experiment='fedmoe', setting=setting, seed=0)


# ---------------------------------------------------------------- A, B, C
if RUN_A_SUPERSET_MOE:
    _moe_block(A_CONFIGS, 'union', 'fedmoe_', 'union',
               'PART A  FedMoE, 25-class space')

if RUN_B_BINARY_MOE:
    _moe_block(B_CONFIGS, 'binary', 'binary_moe_', 'binary',
               'PART B  FedMoE, 2-class space')

if RUN_C_K8_K9:
    _moe_block(C_CONFIGS, 'union', 'fedmoe_', 'union',
               'PART C  K = 8 and K = 9')

# ---------------------------------------------------------------- D: seeds
# Repeats every method at extra seeds, so each one gets its own error bar
# instead of borrowing FedAvg's. This is the change that most increases
# confidence in the thesis, and it costs only compute.
if RUN_D_SEEDS:
    for space in SEED_SPACES:
        cl = _clients(space)
        tag = 'union' if space == 'union' else 'binary'
        print(f'\n{"=" * 70}\nPART D  extra seeds, {space} space, '
              f'seeds {SEEDS_TO_ADD}\n{"=" * 70}')
        for seed in SEEDS_TO_ADD:
            _step(f'centralized seed {seed} ({space})', tag,
                  lambda s=seed: run_centralized(cl, seed=s, setting=f'{tag}_s{s}'),
                  experiment='centralized', setting=f'{tag}_s{seed}', seed=seed)
            _step(f'local_only seed {seed} ({space})', tag,
                  lambda s=seed: run_local_only(cl, seed=s, setting=f'{tag}_s{s}'),
                  experiment='local_only', setting=f'{tag}_s{seed}', seed=seed)
            for algo in ('fedavg', 'fedprox', 'fedadam'):
                _step(f'{algo} seed {seed} ({space})', tag,
                      lambda a=algo, s=seed: run_federated(cl, algo=a, seed=s,
                                                           setting=f'{tag}_s{s}',
                                                           verbose=False),
                      experiment='federated', algo=algo,
                      setting=f'{tag}_s{seed}', seed=seed)
            if 'run_distillation' in globals():
                _step(f'feddistill seed {seed} ({space})', tag,
                      lambda s=seed: run_distillation(cl, seed=s,
                                                      setting=f'{tag}_s{s}',
                                                      verbose=False),
                      experiment='federated', algo='feddistill',
                      setting=f'{tag}_s{seed}', seed=seed)
            if 'run_fedmix' in globals():
                _step(f'fedmix seed {seed} ({space})', tag,
                      lambda s=seed: run_fedmix(cl, lam=0.2, seed=s,
                                                setting=f'{tag}_s{s}'),
                      experiment='fedmix', setting=f'{tag}_s{seed}', seed=seed)

# ---------------------------------------------------------------- summary
print(f'\n{"=" * 70}')
print(f'ran {len(_ran)}, skipped {len(_skipped)} already present, '
      f'{len(RESULTS) - _start} new rows')
print('=' * 70)

rows = [r for r in RESULTS if r.get('experiment') == 'fedmoe' and r.get('seed') == 0]
by = {}
for r in rows:
    by.setdefault(r['setting'], []).append(r)

if by:
    print(f'\nFedMoE, every configuration now on file:\n')
    print(f'{"setting":18s} {"space":<8s} {"K":>3s} {"top":>4s} {"shared":<20s} '
          f'{"acc":>7s} {"used":>7s} {"MB":>7s}')
    for s, rs in sorted(by.items(), key=lambda kv: (kv[1][0].get('label_space', ''),
                                                    kv[1][0]['num_experts'],
                                                    kv[1][0]['top_k'],
                                                    not kv[1][0]['share_backbone'])):
        r = rs[0]
        acc = sum(x['accuracy'] for x in rs) / len(rs)
        mb = (r.get('up_mb') or 0) + (r.get('down_mb') or 0)
        shared = 'backbone + experts' if r['share_backbone'] else 'experts only'
        print(f'{s:18s} {r.get("label_space", "?"):<8s} {r["num_experts"]:3d} '
              f'{r["top_k"]:4d} {shared:<20s} {acc:7.4f} '
              f'{r["experts_used"]:3d}/{r["num_experts"]:<3d} {mb:7.0f}')

print('\nSend results.json back and every thesis table and figure rebuilds from it.')



PART A  FedMoE, 25-class space: 8 configurations

--- fedmoe_K5  (K=5, top-1, backbone + experts) ---
  FedMoE (K=5,top1,bb=1) r01 | mean acc 0.7289 | mean f1 0.5927
  FedMoE (K=5,top1,bb=1) r02 | mean acc 0.6910 | mean f1 0.5300
  FedMoE (K=5,top1,bb=1) r03 | mean acc 0.7195 | mean f1 0.5580
  FedMoE (K=5,top1,bb=1) r04 | mean acc 0.7029 | mean f1 0.5824
  FedMoE (K=5,top1,bb=1) r05 | mean acc 0.7053 | mean f1 0.5432
  FedMoE (K=5,top1,bb=1) r06 | mean acc 0.7152 | mean f1 0.5805
  FedMoE (K=5,top1,bb=1) r07 | mean acc 0.7145 | mean f1 0.5987
  FedMoE (K=5,top1,bb=1) r08 | mean acc 0.7082 | mean f1 0.5830
FedMoE (K=5,top1,bb=1) | mean acc 0.7082 | mean f1 0.5830 | 3676.5 MB | 271s
  experts receiving >=1% of images: 4 of 5 | distinct dominant experts: 3 | mean gate entropy 0.142
  expert usage share over all clients: [0.314, 0.365, 0.196, 0.003, 0.121]
  per-client gate weights (local, personalized, never shared with the server):
    {'client': 'PneumoniaMNIST', 'modality': 'chest X-